# 07 — Transferencia entre sensores y robustez de calibración

## Proyecto
**Kaki_DeepHS_StorageDays**

## Objetivo

Evaluar si la señal temporal aprendida con una cámara HSI puede transferirse a la otra cámara cuando ambas se restringen al rango espectral común.

Se estudian dos direcciones:

\[
VIS\_COR \rightarrow VIS
\]

\[
VIS \rightarrow VIS\_COR
\]

con una grilla común de:

\[
410\text{–}900\ {\rm nm}
\]

cada 5 nm.

## Preguntas

1. ¿Cuánto desempeño se conserva al cambiar de cámara?
2. ¿SNV reduce el dominio instrumental sin calibración explícita?
3. ¿Una calibración espectral por longitud de onda aprendida **solo con frutos de training** mejora la transferencia?
4. ¿La transferencia entre sensores sigue superando un null correcto cuando también se excluye el día/sesión objetivo?
5. ¿Qué parte de la pérdida se debe al cambio de sensor y qué parte a restringir el espectro al intervalo común?

## Protocolos

### A. Within-sensor common-grid

Cada cámara se entrena y prueba consigo misma, pero usando únicamente la grilla 410–900 nm.  
Sirve para medir el costo de restringir y remuestrear el espectro.

### B. Cross-sensor directo

El predictor se entrena en el sensor fuente y se aplica al sensor destino sin usar espectros del destino para calibración.

### C. Cross-sensor con calibración afín pareada

Con los frutos de desarrollo únicamente se aprende, para cada longitud de onda:

\[
x_{\text{source},\lambda}
=
a_\lambda +
b_\lambda x_{\text{target},\lambda}
\]

La calibración usa pares fuente/destino del mismo `fruit_id × storage_day` en training.  
No usa los frutos del outer test.

Después el predictor entrenado en el sensor fuente recibe el espectro destino transformado al dominio fuente.

### D. Cross-sensor session-excluded

Para predecir el día \(d\):

- se excluye \(d\) del ajuste del predictor;
- se excluye \(d\) de la calibración afín;
- el outer test sigue siendo fruit-disjoint;
- se compara contra `SessionExcluded_DummyMean`.

## Interpretación

La transferencia **directa** es la prueba más fuerte de portabilidad instrumental.

La transferencia **calibrada** es más realista cuando se dispone de un pequeño conjunto de muestras medidas con ambos instrumentos, pero requiere datos pareados del sensor destino durante calibración.

Una sola campaña sigue sin constituir validación externa temporal.


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import itertools
import json
import math
import platform
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "07_kaki_cross_sensor_transfer_calibration"
RUN_SIGNATURE = "cross_sensor_common_grid_v1"

ROOT = PROJECT_ROOT

INTERIM = ROOT / "data/interim"
SPLITS_DIR = ROOT / "data/splits"

BASE02 = ROOT / "results/02_kaki_fruit_disjoint_splits_baselines"

RESULTS = ROOT / "results/07_kaki_cross_sensor_transfer_calibration"
FIGURES = ROOT / "figures/07_kaki_cross_sensor_transfer_calibration"
TABLES = ROOT / "tables/07_kaki_cross_sensor_transfer_calibration"
EXPORT_ROOT = ROOT / "exports/07_kaki_cross_sensor_transfer_calibration"
CHECKPOINT_DIR = RESULTS / f"checkpoints_{RUN_SIGNATURE}"

for p in [
    RESULTS,
    FIGURES,
    TABLES,
    EXPORT_ROOT,
    CHECKPOINT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

COMMON_GRID = np.arange(
    410.0,
    900.0 + 0.001,
    5.0
)

PREPROCESSING = [
    "raw",
    "snv",
]

TRANSFER_METHODS = [
    "direct_raw",
    "direct_snv",
    "affine_raw",
    "affine_snv",
]

RIDGE_ALPHAS = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
]

OUTER_FOLDS = 5
INNER_FOLDS = 4

RANDOM_SEED = 20260818
BOOTSTRAP_REPS = 5000

RESUME = True

print("COMMON_GRID:", len(COMMON_GRID), "puntos")
print(COMMON_GRID[0], "→", COMMON_GRID[-1], "nm")
print("RESUME:", RESUME)


## 2. Cargar VIS, VIS_COR y splits congelados


In [ ]:
SPECTRA_PATHS = {
    "VIS":
        INTERIM /
        "kaki_fruit_day_spectra_VIS.csv",

    "VIS_COR":
        INTERIM /
        "kaki_fruit_day_spectra_VIS_COR.csv",
}

CUBE_PATHS = {
    "VIS":
        INTERIM /
        "kaki_cube_spectra_VIS.csv",

    "VIS_COR":
        INTERIM /
        "kaki_cube_spectra_VIS_COR.csv",
}

OUTER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_outer5_fruit_disjoint_v1.csv"
)

INNER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_inner4_within_outer5_fruit_disjoint_v1.csv"
)

BASELINE_OOF_PATH = (
    BASE02 /
    "kaki_oof_predictions.csv"
)

for p in [
    *SPECTRA_PATHS.values(),
    *CUBE_PATHS.values(),
    OUTER_SPLIT_PATH,
    INNER_SPLIT_PATH,
    BASELINE_OOF_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(
            f"Falta archivo requerido: {p}"
        )

expected_cube_counts = {
    "VIS": 395,
    "VIS_COR": 398,
}

spectra = {}
native_wavelengths = {}

gate_rows = []

for camera in [
    "VIS",
    "VIS_COR"
]:

    cube = pd.read_csv(
        CUBE_PATHS[
            camera
        ]
    )

    gate_rows.append({
        "camera_type":
            camera,

        "n_cubes":
            int(
                len(
                    cube
                )
            ),

        "median_mask_coverage":
            float(
                cube[
                    "mask_coverage"
                ].median()
            ),

        "n_mask_qc_not_ok":
            int(
                (
                    cube[
                        "mask_qc_flag"
                    ]
                    !=
                    "ok"
                ).sum()
            ),

        "n_extraction_errors":
            int(
                cube[
                    "extraction_error"
                ].notna().sum()
            ),
    })

    assert len(
        cube
    ) == expected_cube_counts[
        camera
    ]

    assert float(
        cube[
            "mask_coverage"
        ].median()
    ) < 0.87

    assert (
        cube[
            "mask_qc_flag"
        ]
        ==
        "ok"
    ).all()

    assert cube[
        "extraction_error"
    ].isna().all()

    df = pd.read_csv(
        SPECTRA_PATHS[
            camera
        ]
    )

    df[
        "fruit_id"
    ] = pd.to_numeric(
        df[
            "fruit_id"
        ],
        errors="raise"
    ).astype(int)

    df[
        "storage_days_model"
    ] = pd.to_numeric(
        df[
            "storage_days_model"
        ],
        errors="raise"
    ).astype(int)

    wl_cols = [
        c
        for c in df.columns
        if c.startswith(
            "wl_"
        )
    ]

    wl = np.array(
        [
            float(
                c.replace(
                    "wl_",
                    ""
                )
            )
            for c in wl_cols
        ],
        dtype=float
    )

    assert len(
        df
    ) == 199

    assert df[
        "fruit_id"
    ].nunique() == 40

    assert not df[
        wl_cols
    ].isna().any().any()

    spectra[
        camera
    ] = df.sort_values(
        [
            "fruit_id",
            "storage_days_model"
        ]
    ).reset_index(
        drop=True
    )

    native_wavelengths[
        camera
    ] = wl

gate = pd.DataFrame(
    gate_rows
)

display(
    gate
)

gate.to_csv(
    TABLES /
    "01_input_mask_gate.csv",
    index=False
)

outer = pd.read_csv(
    OUTER_SPLIT_PATH
)

inner = pd.read_csv(
    INNER_SPLIT_PATH
)

baseline_oof = pd.read_csv(
    BASELINE_OOF_PATH
)

for df in [
    outer,
    inner,
]:

    df[
        "fruit_id"
    ] = pd.to_numeric(
        df[
            "fruit_id"
        ],
        errors="raise"
    ).astype(int)

outer[
    "outer_fold"
] = pd.to_numeric(
    outer[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "outer_fold"
] = pd.to_numeric(
    inner[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "inner_fold"
] = pd.to_numeric(
    inner[
        "inner_fold"
    ],
    errors="raise"
).astype(int)

print(
    "VIS:",
    len(
        native_wavelengths[
            "VIS"
        ]
    ),
    "bandas |",
    native_wavelengths[
        "VIS"
    ].min(),
    "→",
    native_wavelengths[
        "VIS"
    ].max()
)

print(
    "VIS_COR:",
    len(
        native_wavelengths[
            "VIS_COR"
        ]
    ),
    "bandas |",
    native_wavelengths[
        "VIS_COR"
    ].min(),
    "→",
    native_wavelengths[
        "VIS_COR"
    ].max()
)

print("✅ Inputs cargados.")


## 3. Verificar emparejamiento exacto fruit×day entre cámaras


In [ ]:
KEY = [
    "fruit_id",
    "storage_days_model",
]

keys_vis = set(
    map(
        tuple,
        spectra[
            "VIS"
        ][
            KEY
        ].to_numpy()
    )
)

keys_cor = set(
    map(
        tuple,
        spectra[
            "VIS_COR"
        ][
            KEY
        ].to_numpy()
    )
)

assert keys_vis == keys_cor
assert len(
    keys_vis
) == 199

pair_audit = (
    spectra[
        "VIS"
    ][
        KEY
    ]
    .merge(
        spectra[
            "VIS_COR"
        ][
            KEY
        ],
        on=KEY,
        how="outer",
        indicator=True
    )
)

assert (
    pair_audit[
        "_merge"
    ]
    ==
    "both"
).all()

print(
    "✅ 199/199 fruit-days emparejados entre VIS y VIS_COR."
)


## 4. Auditoría de splits


In [ ]:
all_fruits = set(
    outer[
        "fruit_id"
    ].astype(int)
)

split_rows = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    train_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_fruits = set(
        inner.loc[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    split_rows.append({
        "outer_fold":
            int(
                outer_fold
            ),

        "n_train_fruits":
            len(
                train_fruits
            ),

        "n_test_fruits":
            len(
                test_fruits
            ),

        "train_test_overlap":
            len(
                train_fruits
                &
                test_fruits
            ),

        "inner_test_overlap":
            len(
                inner_fruits
                &
                test_fruits
            ),

        "inner_matches_train":
            inner_fruits
            ==
            train_fruits,
    })

split_audit = pd.DataFrame(
    split_rows
)

display(
    split_audit
)

assert (
    split_audit[
        "train_test_overlap"
    ]
    ==
    0
).all()

assert (
    split_audit[
        "inner_test_overlap"
    ]
    ==
    0
).all()

assert split_audit[
    "inner_matches_train"
].all()

split_audit.to_csv(
    TABLES /
    "02_split_audit.csv",
    index=False
)

print("✅ Sin fuga de identidad.")


# 5. Interpolar ambas cámaras a la grilla común 410–900 nm

La interpolación es lineal y se realiza por cada espectro fruto×día.

No se extrapola fuera del rango nativo de ninguna cámara.


In [ ]:
def interpolate_spectra_to_grid(
    df,
    native_wavelength_grid,
    target_grid
):

    spectral_columns = [
        c
        for c in df.columns
        if c.startswith(
            "wl_"
        )
    ]

    X = df[
        spectral_columns
    ].to_numpy(
        dtype=float
    )

    native_wavelength_grid = np.asarray(
        native_wavelength_grid,
        dtype=float
    )

    target_grid = np.asarray(
        target_grid,
        dtype=float
    )

    if (
        target_grid.min()
        <
        native_wavelength_grid.min()
        or
        target_grid.max()
        >
        native_wavelength_grid.max()
    ):
        raise RuntimeError(
            "La grilla objetivo requiere extrapolación."
        )

    Y = np.empty(
        (
            len(
                df
            ),
            len(
                target_grid
            )
        ),
        dtype=float
    )

    for i in range(
        len(
            df
        )
    ):

        Y[
            i,
            :
        ] = np.interp(
            target_grid,
            native_wavelength_grid,
            X[
                i,
                :
            ]
        )

    out = pd.DataFrame({
        "fruit_id":
            df[
                "fruit_id"
            ].to_numpy(
                dtype=int
            ),

        "storage_days_model":
            df[
                "storage_days_model"
            ].to_numpy(
                dtype=int
            ),

        "day_code":
            df[
                "day_code"
            ].astype(
                str
            ).to_numpy(),
    })

    for j, wl in enumerate(
        target_grid
    ):

        out[
            f"cg_{wl:.1f}"
        ] = Y[
            :,
            j
        ]

    return out


common = {}

for camera in [
    "VIS",
    "VIS_COR"
]:

    common[
        camera
    ] = interpolate_spectra_to_grid(
        spectra[
            camera
        ],
        native_wavelengths[
            camera
        ],
        COMMON_GRID
    )

    assert len(
        common[
            camera
        ]
    ) == 199

    assert common[
        camera
    ][
        "fruit_id"
    ].nunique() == 40

common_cols = [
    c
    for c in common[
        "VIS"
    ].columns
    if c.startswith(
        "cg_"
    )
]

assert len(
    common_cols
) == len(
    COMMON_GRID
)

print(
    "✅ Common grid:",
    len(
        common_cols
    ),
    "features."
)


## 6. Diagnóstico descriptivo de alineación entre cámaras

Se calcula la diferencia absoluta entre VIS y VIS_COR para cada fruit-day sobre la misma grilla.

Esto es descriptivo y no interviene en la selección de modelos.


In [ ]:
paired = (
    common[
        "VIS"
    ]
    .merge(
        common[
            "VIS_COR"
        ],
        on=[
            "fruit_id",
            "storage_days_model",
            "day_code",
        ],
        suffixes=(
            "_VIS",
            "_VIS_COR"
        ),
        how="inner"
    )
)

assert len(
    paired
) == 199

delta_rows = []

for wl in COMMON_GRID:

    a = paired[
        f"cg_{wl:.1f}_VIS"
    ].to_numpy(
        dtype=float
    )

    b = paired[
        f"cg_{wl:.1f}_VIS_COR"
    ].to_numpy(
        dtype=float
    )

    delta_rows.append({
        "wavelength_nm":
            float(
                wl
            ),

        "mean_VIS":
            float(
                np.mean(
                    a
                )
            ),

        "mean_VIS_COR":
            float(
                np.mean(
                    b
                )
            ),

        "mean_abs_difference":
            float(
                np.mean(
                    np.abs(
                        a
                        -
                        b
                    )
                )
            ),

        "pearson_like_correlation":
            float(
                np.corrcoef(
                    a,
                    b
                )[
                    0,
                    1
                ]
            ),
    })

sensor_alignment = pd.DataFrame(
    delta_rows
)

sensor_alignment.to_csv(
    TABLES /
    "03_sensor_alignment_common_grid.csv",
    index=False
)

fig, ax = plt.subplots(
    figsize=(9, 4.8)
)

ax.plot(
    sensor_alignment[
        "wavelength_nm"
    ],
    sensor_alignment[
        "mean_abs_difference"
    ]
)

ax.set_xlabel(
    "Longitud de onda (nm)"
)

ax.set_ylabel(
    "Diferencia absoluta media VIS vs VIS_COR"
)

ax.set_title(
    "Diferencia instrumental sobre la grilla común"
)

ax.grid(
    alpha=0.15
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "01_common_grid_sensor_absolute_difference.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 7. Preprocesamiento y calibración afín por longitud de onda


In [ ]:
def snv(
    X
):

    X = np.asarray(
        X,
        dtype=float
    )

    mu = X.mean(
        axis=1,
        keepdims=True
    )

    sd = X.std(
        axis=1,
        keepdims=True
    )

    sd[
        (~np.isfinite(sd))
        |
        (sd == 0)
    ] = 1.0

    return (
        X
        -
        mu
    ) / sd


def apply_representation(
    X,
    representation
):

    if representation == "raw":
        return np.asarray(
            X,
            dtype=float
        ).copy()

    if representation == "snv":
        return snv(
            X
        )

    raise ValueError(
        representation
    )


def fit_affine_target_to_source(
    X_target,
    X_source
):

    X_target = np.asarray(
        X_target,
        dtype=float
    )

    X_source = np.asarray(
        X_source,
        dtype=float
    )

    if X_target.shape != X_source.shape:
        raise RuntimeError(
            "Affine calibration requires paired matrices "
            "with identical shape."
        )

    mean_t = X_target.mean(
        axis=0
    )

    mean_s = X_source.mean(
        axis=0
    )

    tc = (
        X_target
        -
        mean_t[
            None,
            :
        ]
    )

    sc = (
        X_source
        -
        mean_s[
            None,
            :
        ]
    )

    var_t = np.sum(
        tc ** 2,
        axis=0
    )

    cov_ts = np.sum(
        tc
        *
        sc,
        axis=0
    )

    slope = np.divide(
        cov_ts,
        var_t,
        out=np.zeros_like(
            cov_ts,
            dtype=float
        ),
        where=var_t > 1e-16
    )

    intercept = (
        mean_s
        -
        slope
        *
        mean_t
    )

    return {
        "slope":
            slope,

        "intercept":
            intercept,
    }


def apply_affine(
    X_target,
    affine
):

    return (
        np.asarray(
            X_target,
            dtype=float
        )
        *
        affine[
            "slope"
        ][
            None,
            :
        ]
        +
        affine[
            "intercept"
        ][
            None,
            :
        ]
    )


def fit_predict_ridge(
    X_train,
    y_train,
    X_test,
    alpha
):

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(
        X_train
    )

    X_test_s = scaler.transform(
        X_test
    )

    model = Ridge(
        alpha=float(
            alpha
        )
    )

    model.fit(
        X_train_s,
        y_train
    )

    return np.asarray(
        model.predict(
            X_test_s
        ),
        dtype=float
    )


## 8. Funciones de métricas


In [ ]:
def fruit_balanced_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def day_balanced_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "y_true"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def metric_bundle(
    df
):

    y = df[
        "y_true"
    ].to_numpy(
        dtype=float
    )

    p = df[
        "y_pred"
    ].to_numpy(
        dtype=float
    )

    residual = (
        p
        -
        y
    )

    rho, _ = spearmanr(
        y,
        p
    )

    return {
        "n_observations":
            int(
                len(
                    df
                )
            ),

        "n_fruits":
            int(
                df[
                    "fruit_id"
                ].nunique()
            ),

        "mae":
            float(
                mean_absolute_error(
                    y,
                    p
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y,
                        p
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y,
                    p
                )
            ),

        "spearman_rho":
            float(
                rho
            ),

        "within_1_day_accuracy":
            float(
                np.mean(
                    np.abs(
                        residual
                    )
                    <=
                    1.0
                )
            ),

        "fruit_balanced_mae":
            fruit_balanced_mae(
                df
            ),

        "day_balanced_mae":
            day_balanced_mae(
                df
            ),

        "bias":
            float(
                np.mean(
                    residual
                )
            ),
    }


# 9. Utilidades para emparejar source y target por fruit×day


In [ ]:
def subset_by_fruits(
    camera,
    fruit_ids
):

    return (
        common[
            camera
        ][
            common[
                camera
            ][
                "fruit_id"
            ].isin(
                fruit_ids
            )
        ]
        .sort_values(
            [
                "fruit_id",
                "storage_days_model"
            ]
        )
        .reset_index(
            drop=True
        )
    )


def paired_source_target(
    source_camera,
    target_camera,
    fruit_ids,
    exclude_day=None
):

    src = subset_by_fruits(
        source_camera,
        fruit_ids
    )

    tgt = subset_by_fruits(
        target_camera,
        fruit_ids
    )

    if exclude_day is not None:

        src = src[
            src[
                "storage_days_model"
            ]
            !=
            exclude_day
        ].copy()

        tgt = tgt[
            tgt[
                "storage_days_model"
            ]
            !=
            exclude_day
        ].copy()

    merged = src.merge(
        tgt,
        on=[
            "fruit_id",
            "storage_days_model",
            "day_code",
        ],
        suffixes=(
            "_SOURCE",
            "_TARGET"
        ),
        how="inner"
    )

    assert len(
        merged
    ) == len(
        src
    ) == len(
        tgt
    )

    X_source = np.column_stack(
        [
            merged[
                f"{c}_SOURCE"
            ].to_numpy(
                dtype=float
            )
            for c in common_cols
        ]
    )

    X_target = np.column_stack(
        [
            merged[
                f"{c}_TARGET"
            ].to_numpy(
                dtype=float
            )
            for c in common_cols
        ]
    )

    y = merged[
        "storage_days_model"
    ].to_numpy(
        dtype=float
    )

    fruit_id = merged[
        "fruit_id"
    ].to_numpy(
        dtype=int
    )

    storage_days = merged[
        "storage_days_model"
    ].to_numpy(
        dtype=int
    )

    return {
        "merged":
            merged,

        "X_source":
            X_source,

        "X_target":
            X_target,

        "y":
            y,

        "fruit_id":
            fruit_id,

        "storage_days":
            storage_days,
    }


# 10. CV interna para transferencia estándar

Cada candidato está definido por:

- `direct_raw`
- `direct_snv`
- `affine_raw`
- `affine_snv`

y `Ridge(alpha)`.

En `affine_*`, la calibración target→source se aprende únicamente con los frutos inner-training.


In [ ]:
def evaluate_transfer_config_standard(
    source_camera,
    target_camera,
    development_fruits,
    inner_assignment,
    method,
    alpha
):

    representation = (
        "snv"
        if method.endswith(
            "snv"
        )
        else "raw"
    )

    use_affine = method.startswith(
        "affine"
    )

    pred_parts = []

    for inner_fold in range(
        INNER_FOLDS
    ):

        val_fruits = set(
            inner_assignment.loc[
                inner_assignment[
                    "inner_fold"
                ]
                ==
                inner_fold,
                "fruit_id"
            ].astype(int)
        )

        train_fruits = (
            set(
                development_fruits
            )
            -
            val_fruits
        )

        train_pair = paired_source_target(
            source_camera,
            target_camera,
            train_fruits
        )

        val_pair = paired_source_target(
            source_camera,
            target_camera,
            val_fruits
        )

        Xs_train = apply_representation(
            train_pair[
                "X_source"
            ],
            representation
        )

        Xt_train = apply_representation(
            train_pair[
                "X_target"
            ],
            representation
        )

        Xt_val = apply_representation(
            val_pair[
                "X_target"
            ],
            representation
        )

        if use_affine:

            calibration = fit_affine_target_to_source(
                Xt_train,
                Xs_train
            )

            Xt_val_used = apply_affine(
                Xt_val,
                calibration
            )

        else:

            Xt_val_used = Xt_val

        pred = fit_predict_ridge(
            Xs_train,
            train_pair[
                "y"
            ],
            Xt_val_used,
            alpha=alpha
        )

        pred_parts.append(
            pd.DataFrame({
                "fruit_id":
                    val_pair[
                        "fruit_id"
                    ],

                "y_true":
                    val_pair[
                        "y"
                    ],

                "y_pred":
                    pred,
            })
        )

    pred_df = pd.concat(
        pred_parts,
        ignore_index=True
    )

    return {
        "fruit_balanced_mae":
            fruit_balanced_mae(
                pred_df
            ),

        "day_balanced_mae":
            day_balanced_mae(
                pred_df
            ),

        "mae":
            float(
                mean_absolute_error(
                    pred_df[
                        "y_true"
                    ],
                    pred_df[
                        "y_pred"
                    ]
                )
            ),
    }


def inner_select_transfer_standard(
    source_camera,
    target_camera,
    development_fruits,
    inner_assignment
):

    rows = []

    best = None

    for method, alpha in itertools.product(
        TRANSFER_METHODS,
        RIDGE_ALPHAS
    ):

        score = evaluate_transfer_config_standard(
            source_camera,
            target_camera,
            development_fruits,
            inner_assignment,
            method,
            alpha
        )

        rows.append({
            "source_camera":
                source_camera,

            "target_camera":
                target_camera,

            "method":
                method,

            "alpha":
                float(
                    alpha
                ),

            **score,
        })

        rank = (
            score[
                "fruit_balanced_mae"
            ],
            score[
                "day_balanced_mae"
            ],
            score[
                "mae"
            ],
            method,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "method":
                    method,

                "alpha":
                    float(
                        alpha
                    ),

                **score,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 11. OOF cross-sensor estándar


In [ ]:
DIRECTIONS = [
    (
        "VIS_COR",
        "VIS"
    ),
    (
        "VIS",
        "VIS_COR"
    ),
]

cross_standard_parts = []
cross_standard_selected_rows = []
cross_standard_search_parts = []
affine_diagnostic_rows = []

for source_camera, target_camera in DIRECTIONS:

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer.loc[
                outer[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        development_fruits = (
            all_fruits
            -
            test_fruits
        )

        inner_assignment = inner[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold
        ][
            [
                "fruit_id",
                "inner_fold"
            ]
        ].copy()

        print(
            f"{source_camera} → {target_camera} | "
            f"outer {outer_fold}"
        )

        best, search_df = (
            inner_select_transfer_standard(
                source_camera,
                target_camera,
                development_fruits,
                inner_assignment
            )
        )

        representation = (
            "snv"
            if best[
                "method"
            ].endswith(
                "snv"
            )
            else "raw"
        )

        use_affine = best[
            "method"
        ].startswith(
            "affine"
        )

        train_pair = paired_source_target(
            source_camera,
            target_camera,
            development_fruits
        )

        test_pair = paired_source_target(
            source_camera,
            target_camera,
            test_fruits
        )

        Xs_train = apply_representation(
            train_pair[
                "X_source"
            ],
            representation
        )

        Xt_train = apply_representation(
            train_pair[
                "X_target"
            ],
            representation
        )

        Xt_test = apply_representation(
            test_pair[
                "X_target"
            ],
            representation
        )

        if use_affine:

            calibration = fit_affine_target_to_source(
                Xt_train,
                Xs_train
            )

            Xt_test_used = apply_affine(
                Xt_test,
                calibration
            )

            affine_diagnostic_rows.append({
                "source_camera":
                    source_camera,

                "target_camera":
                    target_camera,

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "method":
                    best[
                        "method"
                    ],

                "median_abs_slope":
                    float(
                        np.median(
                            np.abs(
                                calibration[
                                    "slope"
                                ]
                            )
                        )
                    ),

                "median_abs_intercept":
                    float(
                        np.median(
                            np.abs(
                                calibration[
                                    "intercept"
                                ]
                            )
                        )
                    ),

                "train_alignment_mae_before":
                    float(
                        np.mean(
                            np.abs(
                                Xt_train
                                -
                                Xs_train
                            )
                        )
                    ),

                "test_alignment_mae_after":
                    float(
                        np.mean(
                            np.abs(
                                Xt_test_used
                                -
                                apply_representation(
                                    test_pair[
                                        "X_source"
                                    ],
                                    representation
                                )
                            )
                        )
                    ),
            })

        else:

            Xt_test_used = Xt_test

        pred = fit_predict_ridge(
            Xs_train,
            train_pair[
                "y"
            ],
            Xt_test_used,
            alpha=best[
                "alpha"
            ]
        )

        cross_standard_parts.append(
            pd.DataFrame({
                "source_camera":
                    source_camera,

                "target_camera":
                    target_camera,

                "protocol":
                    "cross_sensor_standard",

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "method":
                    best[
                        "method"
                    ],

                "alpha":
                    float(
                        best[
                            "alpha"
                        ]
                    ),

                "fruit_id":
                    test_pair[
                        "fruit_id"
                    ],

                "storage_days":
                    test_pair[
                        "storage_days"
                    ],

                "y_true":
                    test_pair[
                        "y"
                    ],

                "y_pred":
                    pred,
            })
        )

        cross_standard_selected_rows.append({
            "source_camera":
                source_camera,

            "target_camera":
                target_camera,

            "outer_fold":
                int(
                    outer_fold
                ),

            "method":
                best[
                    "method"
                ],

            "alpha":
                float(
                    best[
                        "alpha"
                    ]
                ),

            "inner_fruit_balanced_mae":
                float(
                    best[
                        "fruit_balanced_mae"
                    ]
                ),

            "inner_day_balanced_mae":
                float(
                    best[
                        "day_balanced_mae"
                    ]
                ),
        })

        search_df[
            "outer_fold"
        ] = int(
            outer_fold
        )

        cross_standard_search_parts.append(
            search_df
        )

cross_standard_oof = pd.concat(
    cross_standard_parts,
    ignore_index=True
)

cross_standard_selected = pd.DataFrame(
    cross_standard_selected_rows
)

cross_standard_search = pd.concat(
    cross_standard_search_parts,
    ignore_index=True
)

affine_diagnostics = pd.DataFrame(
    affine_diagnostic_rows
)

for source_camera, target_camera in DIRECTIONS:

    g = cross_standard_oof[
        (
            cross_standard_oof[
                "source_camera"
            ]
            ==
            source_camera
        )
        &
        (
            cross_standard_oof[
                "target_camera"
            ]
            ==
            target_camera
        )
    ]

    assert len(
        g
    ) == 199

    assert g[
        "fruit_id"
    ].nunique() == 40

print("✅ Cross-sensor estándar completo.")


## 12. Métricas cross-sensor estándar


In [ ]:
cross_standard_metric_rows = []

for (
    source_camera,
    target_camera
), g in cross_standard_oof.groupby(
    [
        "source_camera",
        "target_camera"
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "source_camera":
            source_camera,

        "target_camera":
            target_camera,

        "protocol":
            "cross_sensor_standard",
    })

    cross_standard_metric_rows.append(
        m
    )

cross_standard_metrics = pd.DataFrame(
    cross_standard_metric_rows
)

display(
    cross_standard_metrics
)

display(
    cross_standard_selected
)

cross_standard_metrics.to_csv(
    TABLES /
    "04_cross_sensor_standard_metrics.csv",
    index=False
)

cross_standard_selected.to_csv(
    TABLES /
    "05_cross_sensor_standard_selected_configs.csv",
    index=False
)

cross_standard_search.to_csv(
    TABLES /
    "06_cross_sensor_standard_inner_search.csv",
    index=False
)

affine_diagnostics.to_csv(
    TABLES /
    "07_affine_calibration_diagnostics.csv",
    index=False
)


# 13. Within-sensor common-grid nested baselines

Se entrena y prueba cada cámara consigo misma sobre las mismas 99 variables comunes.

Esto cuantifica cuánto perdemos simplemente por:

- eliminar 900–1004 nm de VIS;
- eliminar 408–410 nm de VIS_COR;
- remuestrear ambos sensores cada 5 nm.


In [ ]:
def inner_select_within_sensor(
    camera,
    development_fruits,
    inner_assignment
):

    rows = []

    best = None

    df = common[
        camera
    ]

    for representation, alpha in itertools.product(
        PREPROCESSING,
        RIDGE_ALPHAS
    ):

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            train_fruits = (
                set(
                    development_fruits
                )
                -
                val_fruits
            )

            tr = df[
                df[
                    "fruit_id"
                ].isin(
                    train_fruits
                )
            ]

            va = df[
                df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            Xtr = apply_representation(
                tr[
                    common_cols
                ].to_numpy(
                    dtype=float
                ),
                representation
            )

            Xva = apply_representation(
                va[
                    common_cols
                ].to_numpy(
                    dtype=float
                ),
                representation
            )

            pred = fit_predict_ridge(
                Xtr,
                tr[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                Xva,
                alpha
            )

            pred_parts.append(
                pd.DataFrame({
                    "fruit_id":
                        va[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        va[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        p = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            p
        )

        db = day_balanced_mae(
            p
        )

        mae = float(
            mean_absolute_error(
                p[
                    "y_true"
                ],
                p[
                    "y_pred"
                ]
            )
        )

        rows.append({
            "camera_type":
                camera,

            "representation":
                representation,

            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        rank = (
            fb,
            db,
            mae,
            representation,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "representation":
                    representation,

                "alpha":
                    float(
                        alpha
                    ),

                "inner_fruit_balanced_mae":
                    fb,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


within_parts = []
within_selected_rows = []

for camera in [
    "VIS",
    "VIS_COR"
]:

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer.loc[
                outer[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        development_fruits = (
            all_fruits
            -
            test_fruits
        )

        inner_assignment = inner[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold
        ][
            [
                "fruit_id",
                "inner_fold"
            ]
        ].copy()

        best, _ = inner_select_within_sensor(
            camera,
            development_fruits,
            inner_assignment
        )

        df = common[
            camera
        ]

        tr = df[
            df[
                "fruit_id"
            ].isin(
                development_fruits
            )
        ]

        te = df[
            df[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ]

        Xtr = apply_representation(
            tr[
                common_cols
            ].to_numpy(
                dtype=float
            ),
            best[
                "representation"
            ]
        )

        Xte = apply_representation(
            te[
                common_cols
            ].to_numpy(
                dtype=float
            ),
            best[
                "representation"
            ]
        )

        pred = fit_predict_ridge(
            Xtr,
            tr[
                "storage_days_model"
            ].to_numpy(
                dtype=float
            ),
            Xte,
            best[
                "alpha"
            ]
        )

        within_parts.append(
            pd.DataFrame({
                "camera_type":
                    camera,

                "protocol":
                    "within_sensor_common_grid",

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "representation":
                    best[
                        "representation"
                    ],

                "alpha":
                    float(
                        best[
                            "alpha"
                        ]
                    ),

                "fruit_id":
                    te[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    pred,
            })
        )

        within_selected_rows.append({
            "camera_type":
                camera,

            "outer_fold":
                int(
                    outer_fold
                ),

            "representation":
                best[
                    "representation"
                ],

            "alpha":
                float(
                    best[
                        "alpha"
                    ]
                ),

            "inner_fruit_balanced_mae":
                float(
                    best[
                        "inner_fruit_balanced_mae"
                    ]
                ),
        })

within_oof = pd.concat(
    within_parts,
    ignore_index=True
)

within_selected = pd.DataFrame(
    within_selected_rows
)

within_metric_rows = []

for camera, g in within_oof.groupby(
    "camera_type"
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            camera,

        "protocol":
            "within_sensor_common_grid",
    })

    within_metric_rows.append(
        m
    )

within_metrics = pd.DataFrame(
    within_metric_rows
)

display(
    within_metrics
)

within_metrics.to_csv(
    TABLES /
    "08_within_sensor_common_grid_metrics.csv",
    index=False
)

within_selected.to_csv(
    TABLES /
    "09_within_sensor_common_grid_selected_configs.csv",
    index=False
)


# 14. Null estándar fruit-disjoint

Se reutiliza el `DummyMedian` del notebook 02.


In [ ]:
dummy_viscor = baseline_oof[
    (
        baseline_oof[
            "camera_type"
        ]
        ==
        "VIS_COR"
    )
    &
    (
        baseline_oof[
            "model"
        ]
        ==
        "DummyMedian"
    )
].copy()

assert len(
    dummy_viscor
) == 199

dummy_standard_metrics = metric_bundle(
    dummy_viscor
)

print(
    "DummyMedian standard FB-MAE:",
    dummy_standard_metrics[
        "fruit_balanced_mae"
    ]
)


# 15. CV interna para transferencia session-excluded

En cada inner fold y día objetivo \(d\):

- source train excluye \(d\);
- target train usado para calibración excluye \(d\);
- target validation contiene exclusivamente \(d\).

Así el día objetivo no participa ni en predictor ni en calibrador.


In [ ]:
def evaluate_transfer_config_session_excluded(
    source_camera,
    target_camera,
    development_fruits,
    inner_assignment,
    method,
    alpha
):

    representation = (
        "snv"
        if method.endswith(
            "snv"
        )
        else "raw"
    )

    use_affine = method.startswith(
        "affine"
    )

    pred_parts = []

    for inner_fold in range(
        INNER_FOLDS
    ):

        val_fruits = set(
            inner_assignment.loc[
                inner_assignment[
                    "inner_fold"
                ]
                ==
                inner_fold,
                "fruit_id"
            ].astype(int)
        )

        train_fruits = (
            set(
                development_fruits
            )
            -
            val_fruits
        )

        val_all = paired_source_target(
            source_camera,
            target_camera,
            val_fruits
        )

        for target_day in sorted(
            np.unique(
                val_all[
                    "storage_days"
                ]
            )
        ):

            train_pair = paired_source_target(
                source_camera,
                target_camera,
                train_fruits,
                exclude_day=int(
                    target_day
                )
            )

            val_day_df = val_all[
                "merged"
            ][
                val_all[
                    "merged"
                ][
                    "storage_days_model"
                ]
                ==
                target_day
            ].copy()

            Xs_train = apply_representation(
                train_pair[
                    "X_source"
                ],
                representation
            )

            Xt_train = apply_representation(
                train_pair[
                    "X_target"
                ],
                representation
            )

            X_target_val = np.column_stack(
                [
                    val_day_df[
                        f"{c}_TARGET"
                    ].to_numpy(
                        dtype=float
                    )
                    for c in common_cols
                ]
            )

            Xt_val = apply_representation(
                X_target_val,
                representation
            )

            if use_affine:

                calibration = fit_affine_target_to_source(
                    Xt_train,
                    Xs_train
                )

                Xt_val_used = apply_affine(
                    Xt_val,
                    calibration
                )

            else:

                Xt_val_used = Xt_val

            pred = fit_predict_ridge(
                Xs_train,
                train_pair[
                    "y"
                ],
                Xt_val_used,
                alpha=alpha
            )

            pred_parts.append(
                pd.DataFrame({
                    "fruit_id":
                        val_day_df[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        val_day_df[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

    p = pd.concat(
        pred_parts,
        ignore_index=True
    )

    return {
        "fruit_balanced_mae":
            fruit_balanced_mae(
                p
            ),

        "day_balanced_mae":
            day_balanced_mae(
                p
            ),

        "mae":
            float(
                mean_absolute_error(
                    p[
                        "y_true"
                    ],
                    p[
                        "y_pred"
                    ]
                )
            ),
    }


def inner_select_transfer_session_excluded(
    source_camera,
    target_camera,
    development_fruits,
    inner_assignment
):

    rows = []

    best = None

    for method, alpha in itertools.product(
        TRANSFER_METHODS,
        RIDGE_ALPHAS
    ):

        score = (
            evaluate_transfer_config_session_excluded(
                source_camera,
                target_camera,
                development_fruits,
                inner_assignment,
                method,
                alpha
            )
        )

        rows.append({
            "source_camera":
                source_camera,

            "target_camera":
                target_camera,

            "method":
                method,

            "alpha":
                float(
                    alpha
                ),

            **score,
        })

        rank = (
            score[
                "fruit_balanced_mae"
            ],
            score[
                "day_balanced_mae"
            ],
            score[
                "mae"
            ],
            method,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "method":
                    method,

                "alpha":
                    float(
                        alpha
                    ),

                **score,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 16. OOF cross-sensor session-excluded


In [ ]:
cross_session_parts = []
cross_session_selected_rows = []
cross_session_search_parts = []

for source_camera, target_camera in DIRECTIONS:

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer.loc[
                outer[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        development_fruits = (
            all_fruits
            -
            test_fruits
        )

        inner_assignment = inner[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold
        ][
            [
                "fruit_id",
                "inner_fold"
            ]
        ].copy()

        print(
            f"SESSION EXCL. {source_camera} → "
            f"{target_camera} | outer {outer_fold}"
        )

        best, search_df = (
            inner_select_transfer_session_excluded(
                source_camera,
                target_camera,
                development_fruits,
                inner_assignment
            )
        )

        representation = (
            "snv"
            if best[
                "method"
            ].endswith(
                "snv"
            )
            else "raw"
        )

        use_affine = best[
            "method"
        ].startswith(
            "affine"
        )

        test_all = paired_source_target(
            source_camera,
            target_camera,
            test_fruits
        )

        for target_day in sorted(
            np.unique(
                test_all[
                    "storage_days"
                ]
            )
        ):

            train_pair = paired_source_target(
                source_camera,
                target_camera,
                development_fruits,
                exclude_day=int(
                    target_day
                )
            )

            test_day_df = test_all[
                "merged"
            ][
                test_all[
                    "merged"
                ][
                    "storage_days_model"
                ]
                ==
                target_day
            ].copy()

            Xs_train = apply_representation(
                train_pair[
                    "X_source"
                ],
                representation
            )

            Xt_train = apply_representation(
                train_pair[
                    "X_target"
                ],
                representation
            )

            X_target_test = np.column_stack(
                [
                    test_day_df[
                        f"{c}_TARGET"
                    ].to_numpy(
                        dtype=float
                    )
                    for c in common_cols
                ]
            )

            Xt_test = apply_representation(
                X_target_test,
                representation
            )

            if use_affine:

                calibration = fit_affine_target_to_source(
                    Xt_train,
                    Xs_train
                )

                Xt_test_used = apply_affine(
                    Xt_test,
                    calibration
                )

            else:

                Xt_test_used = Xt_test

            pred = fit_predict_ridge(
                Xs_train,
                train_pair[
                    "y"
                ],
                Xt_test_used,
                alpha=best[
                    "alpha"
                ]
            )

            cross_session_parts.append(
                pd.DataFrame({
                    "source_camera":
                        source_camera,

                    "target_camera":
                        target_camera,

                    "protocol":
                        "cross_sensor_session_excluded",

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "method":
                        best[
                            "method"
                        ],

                    "alpha":
                        float(
                            best[
                                "alpha"
                            ]
                        ),

                    "fruit_id":
                        test_day_df[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        test_day_df[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        test_day_df[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        cross_session_selected_rows.append({
            "source_camera":
                source_camera,

            "target_camera":
                target_camera,

            "outer_fold":
                int(
                    outer_fold
                ),

            "method":
                best[
                    "method"
                ],

            "alpha":
                float(
                    best[
                        "alpha"
                    ]
                ),

            "inner_fruit_balanced_mae":
                float(
                    best[
                        "fruit_balanced_mae"
                    ]
                ),

            "inner_day_balanced_mae":
                float(
                    best[
                        "day_balanced_mae"
                    ]
                ),
        })

        search_df[
            "outer_fold"
        ] = int(
            outer_fold
        )

        cross_session_search_parts.append(
            search_df
        )

cross_session_oof = pd.concat(
    cross_session_parts,
    ignore_index=True
)

cross_session_selected = pd.DataFrame(
    cross_session_selected_rows
)

cross_session_search = pd.concat(
    cross_session_search_parts,
    ignore_index=True
)

for source_camera, target_camera in DIRECTIONS:

    g = cross_session_oof[
        (
            cross_session_oof[
                "source_camera"
            ]
            ==
            source_camera
        )
        &
        (
            cross_session_oof[
                "target_camera"
            ]
            ==
            target_camera
        )
    ]

    assert len(
        g
    ) == 199

    assert g[
        "fruit_id"
    ].nunique() == 40

print("✅ Cross-sensor session-excluded completo.")


## 17. Métricas cross-sensor session-excluded


In [ ]:
cross_session_metric_rows = []

for (
    source_camera,
    target_camera
), g in cross_session_oof.groupby(
    [
        "source_camera",
        "target_camera"
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "source_camera":
            source_camera,

        "target_camera":
            target_camera,

        "protocol":
            "cross_sensor_session_excluded",
    })

    cross_session_metric_rows.append(
        m
    )

cross_session_metrics = pd.DataFrame(
    cross_session_metric_rows
)

display(
    cross_session_metrics
)

display(
    cross_session_selected
)

cross_session_metrics.to_csv(
    TABLES /
    "10_cross_sensor_session_excluded_metrics.csv",
    index=False
)

cross_session_selected.to_csv(
    TABLES /
    "11_cross_sensor_session_excluded_selected_configs.csv",
    index=False
)

cross_session_search.to_csv(
    TABLES /
    "12_cross_sensor_session_excluded_inner_search.csv",
    index=False
)


# 18. Null correcto session-excluded


In [ ]:
session_null_parts = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    development_df = common[
        "VIS_COR"
    ][
        ~common[
            "VIS_COR"
        ][
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    test_df = common[
        "VIS_COR"
    ][
        common[
            "VIS_COR"
        ][
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    for target_day in sorted(
        test_df[
            "storage_days_model"
        ].unique()
    ):

        tr = development_df[
            development_df[
                "storage_days_model"
            ]
            !=
            target_day
        ]

        te = test_df[
            test_df[
                "storage_days_model"
            ]
            ==
            target_day
        ]

        null_value = float(
            tr[
                "storage_days_model"
            ].mean()
        )

        session_null_parts.append(
            pd.DataFrame({
                "outer_fold":
                    int(
                        outer_fold
                    ),

                "fruit_id":
                    te[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    np.repeat(
                        null_value,
                        len(
                            te
                        )
                    ),
            })
        )

session_null_oof = pd.concat(
    session_null_parts,
    ignore_index=True
)

session_null_metrics = metric_bundle(
    session_null_oof
)

print(
    "SessionExcluded_DummyMean FB-MAE:",
    session_null_metrics[
        "fruit_balanced_mae"
    ]
)


# 19. Referencias full-spectrum del notebook 02


In [ ]:
full_reference_rows = []

for camera in [
    "VIS",
    "VIS_COR"
]:

    g = baseline_oof[
        (
            baseline_oof[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            baseline_oof[
                "model"
            ]
            ==
            "Ridge"
        )
    ].copy()

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            camera,

        "reference":
            "full_native_spectrum_Ridge_notebook02",
    })

    full_reference_rows.append(
        m
    )

full_reference = pd.DataFrame(
    full_reference_rows
)

display(
    full_reference
)

full_reference.to_csv(
    TABLES /
    "13_full_native_reference_metrics.csv",
    index=False
)


# 20. Comparación maestra


In [ ]:
comparison_rows = []

for _, row in full_reference.iterrows():

    comparison_rows.append({
        "protocol":
            "full_native_reference",

        "source_camera":
            row[
                "camera_type"
            ],

        "target_camera":
            row[
                "camera_type"
            ],

        "method":
            "native_full_Ridge",

        "fruit_balanced_mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),

        "mae":
            float(
                row[
                    "mae"
                ]
            ),

        "rmse":
            float(
                row[
                    "rmse"
                ]
            ),

        "r2":
            float(
                row[
                    "r2"
                ]
            ),

        "spearman_rho":
            float(
                row[
                    "spearman_rho"
                ]
            ),

        "within_1_day_accuracy":
            float(
                row[
                    "within_1_day_accuracy"
                ]
            ),
    })

for _, row in within_metrics.iterrows():

    comparison_rows.append({
        "protocol":
            "within_sensor_common_grid",

        "source_camera":
            row[
                "camera_type"
            ],

        "target_camera":
            row[
                "camera_type"
            ],

        "method":
            "common_grid_Ridge",

        "fruit_balanced_mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),

        "mae":
            float(
                row[
                    "mae"
                ]
            ),

        "rmse":
            float(
                row[
                    "rmse"
                ]
            ),

        "r2":
            float(
                row[
                    "r2"
                ]
            ),

        "spearman_rho":
            float(
                row[
                    "spearman_rho"
                ]
            ),

        "within_1_day_accuracy":
            float(
                row[
                    "within_1_day_accuracy"
                ]
            ),
    })

for _, row in cross_standard_metrics.iterrows():

    comparison_rows.append({
        "protocol":
            "cross_sensor_standard",

        "source_camera":
            row[
                "source_camera"
            ],

        "target_camera":
            row[
                "target_camera"
            ],

        "method":
            "nested_selected_transfer",

        "fruit_balanced_mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),

        "mae":
            float(
                row[
                    "mae"
                ]
            ),

        "rmse":
            float(
                row[
                    "rmse"
                ]
            ),

        "r2":
            float(
                row[
                    "r2"
                ]
            ),

        "spearman_rho":
            float(
                row[
                    "spearman_rho"
                ]
            ),

        "within_1_day_accuracy":
            float(
                row[
                    "within_1_day_accuracy"
                ]
            ),
    })

for _, row in cross_session_metrics.iterrows():

    comparison_rows.append({
        "protocol":
            "cross_sensor_session_excluded",

        "source_camera":
            row[
                "source_camera"
            ],

        "target_camera":
            row[
                "target_camera"
            ],

        "method":
            "nested_selected_transfer",

        "fruit_balanced_mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),

        "mae":
            float(
                row[
                    "mae"
                ]
            ),

        "rmse":
            float(
                row[
                    "rmse"
                ]
            ),

        "r2":
            float(
                row[
                    "r2"
                ]
            ),

        "spearman_rho":
            float(
                row[
                    "spearman_rho"
                ]
            ),

        "within_1_day_accuracy":
            float(
                row[
                    "within_1_day_accuracy"
                ]
            ),
    })

comparison_metrics = pd.DataFrame(
    comparison_rows
)

display(
    comparison_metrics
)

comparison_metrics.to_csv(
    TABLES /
    "14_master_cross_sensor_comparison.csv",
    index=False
)


# 21. Bootstrap pareado por fruto

Comparaciones principales:

1. Cross-sensor estándar vs within-target common-grid.
2. Cross-sensor estándar vs DummyMedian.
3. Cross-sensor session-excluded vs null correcto.
4. Dirección VIS_COR→VIS vs VIS→VIS_COR.


In [ ]:
def per_fruit_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return (
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
    )


def paired_bootstrap(
    a,
    b,
    label,
    n_boot=5000,
    seed=20260818
):

    ids = sorted(
        set(
            a.index
        )
        &
        set(
            b.index
        )
    )

    av = a.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    bv = b.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    diff = (
        av
        -
        bv
    )

    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot,
        dtype=float
    )

    for i in range(
        n_boot
    ):

        idx = rng.integers(
            0,
            len(
                diff
            ),
            size=len(
                diff
            )
        )

        boot[
            i
        ] = np.mean(
            diff[
                idx
            ]
        )

    return {
        "comparison":
            label,

        "n_fruits":
            int(
                len(
                    diff
                )
            ),

        "mean_delta_mae_days":
            float(
                diff.mean()
            ),

        "ci95_low":
            float(
                np.percentile(
                    boot,
                    2.5
                )
            ),

        "ci95_high":
            float(
                np.percentile(
                    boot,
                    97.5
                )
            ),

        "fraction_first_better":
            float(
                np.mean(
                    diff
                    <
                    0
                )
            ),
    }


bootstrap_rows = []

dummy_pf = per_fruit_mae(
    dummy_viscor
)

session_null_pf = per_fruit_mae(
    session_null_oof
)

seed_offset = 0

for source_camera, target_camera in DIRECTIONS:

    cross_std = cross_standard_oof[
        (
            cross_standard_oof[
                "source_camera"
            ]
            ==
            source_camera
        )
        &
        (
            cross_standard_oof[
                "target_camera"
            ]
            ==
            target_camera
        )
    ]

    within_target = within_oof[
        within_oof[
            "camera_type"
        ]
        ==
        target_camera
    ]

    bootstrap_rows.append(
        paired_bootstrap(
            per_fruit_mae(
                cross_std
            ),
            per_fruit_mae(
                within_target
            ),
            (
                f"{source_camera}->{target_camera} "
                "cross standard minus within-target common-grid"
            ),
            BOOTSTRAP_REPS,
            RANDOM_SEED + seed_offset
        )
    )

    seed_offset += 1

    bootstrap_rows.append(
        paired_bootstrap(
            per_fruit_mae(
                cross_std
            ),
            dummy_pf,
            (
                f"{source_camera}->{target_camera} "
                "cross standard minus DummyMedian"
            ),
            BOOTSTRAP_REPS,
            RANDOM_SEED + seed_offset
        )
    )

    seed_offset += 1

    cross_sess = cross_session_oof[
        (
            cross_session_oof[
                "source_camera"
            ]
            ==
            source_camera
        )
        &
        (
            cross_session_oof[
                "target_camera"
            ]
            ==
            target_camera
        )
    ]

    bootstrap_rows.append(
        paired_bootstrap(
            per_fruit_mae(
                cross_sess
            ),
            session_null_pf,
            (
                f"{source_camera}->{target_camera} "
                "session-excluded minus correct null"
            ),
            BOOTSTRAP_REPS,
            RANDOM_SEED + seed_offset
        )
    )

    seed_offset += 1


a = cross_standard_oof[
    (
        cross_standard_oof[
            "source_camera"
        ]
        ==
        "VIS_COR"
    )
    &
    (
        cross_standard_oof[
            "target_camera"
        ]
        ==
        "VIS"
    )
]

b = cross_standard_oof[
    (
        cross_standard_oof[
            "source_camera"
        ]
        ==
        "VIS"
    )
    &
    (
        cross_standard_oof[
            "target_camera"
        ]
        ==
        "VIS_COR"
    )
]

bootstrap_rows.append(
    paired_bootstrap(
        per_fruit_mae(
            a
        ),
        per_fruit_mae(
            b
        ),
        (
            "VIS_COR->VIS minus VIS->VIS_COR "
            "cross-standard"
        ),
        BOOTSTRAP_REPS,
        RANDOM_SEED + 100
    )
)

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_results
)

bootstrap_results.to_csv(
    TABLES /
    "15_paired_fruit_bootstrap_cross_sensor.csv",
    index=False
)


# 22. Estabilidad de método seleccionado

Permite ver si la transferencia depende sistemáticamente de:

- calibración afín;
- SNV;
- RAW.


In [ ]:
standard_method_stability = (
    cross_standard_selected.groupby(
        [
            "source_camera",
            "target_camera",
            "method"
        ]
    )
    .size()
    .reset_index(
        name="n_outer_folds_selected"
    )
    .sort_values(
        [
            "source_camera",
            "target_camera",
            "n_outer_folds_selected"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

session_method_stability = (
    cross_session_selected.groupby(
        [
            "source_camera",
            "target_camera",
            "method"
        ]
    )
    .size()
    .reset_index(
        name="n_outer_folds_selected"
    )
    .sort_values(
        [
            "source_camera",
            "target_camera",
            "n_outer_folds_selected"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

display(
    standard_method_stability
)

display(
    session_method_stability
)

standard_method_stability.to_csv(
    TABLES /
    "16_cross_sensor_standard_method_stability.csv",
    index=False
)

session_method_stability.to_csv(
    TABLES /
    "17_cross_sensor_session_method_stability.csv",
    index=False
)


# 23. Error y bias por día


In [ ]:
day_rows = []

for protocol, pred_df in [
    (
        "cross_standard",
        cross_standard_oof
    ),
    (
        "cross_session_excluded",
        cross_session_oof
    ),
]:

    for (
        source_camera,
        target_camera,
        day
    ), g in pred_df.groupby(
        [
            "source_camera",
            "target_camera",
            "storage_days"
        ]
    ):

        err = (
            g[
                "y_pred"
            ].to_numpy(
                dtype=float
            )
            -
            g[
                "y_true"
            ].to_numpy(
                dtype=float
            )
        )

        day_rows.append({
            "protocol":
                protocol,

            "source_camera":
                source_camera,

            "target_camera":
                target_camera,

            "storage_days":
                int(
                    day
                ),

            "n":
                int(
                    len(
                        g
                    )
                ),

            "mae":
                float(
                    np.mean(
                        np.abs(
                            err
                        )
                    )
                ),

            "bias":
                float(
                    np.mean(
                        err
                    )
                ),
        })

day_metrics = pd.DataFrame(
    day_rows
)

day_metrics.to_csv(
    TABLES /
    "18_cross_sensor_error_bias_by_day.csv",
    index=False
)

display(
    day_metrics
)


## 24. Figura — desempeño estándar por dominio


In [ ]:
plot_rows = []

for _, row in within_metrics.iterrows():

    plot_rows.append({
        "label":
            f"{row['camera_type']}→{row['camera_type']}\ncommon-grid",

        "mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),
    })

for _, row in cross_standard_metrics.iterrows():

    plot_rows.append({
        "label":
            (
                f"{row['source_camera']}→"
                f"{row['target_camera']}\ncross"
            ),

        "mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),
    })

plot_df = pd.DataFrame(
    plot_rows
)

fig, ax = plt.subplots(
    figsize=(8.5, 4.8)
)

ax.bar(
    plot_df[
        "label"
    ],
    plot_df[
        "mae"
    ]
)

ax.axhline(
    dummy_standard_metrics[
        "fruit_balanced_mae"
    ],
    linestyle="--",
    label="DummyMedian"
)

ax.set_ylabel(
    "MAE fruit-balanced (días)"
)

ax.set_title(
    "Transferencia estándar entre cámaras"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.2
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "02_cross_sensor_standard_mae.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 25. Figura — session-excluded contra null correcto


In [ ]:
fig, ax = plt.subplots(
    figsize=(7.5, 4.8)
)

labels = []

values = []

for _, row in cross_session_metrics.iterrows():

    labels.append(
        (
            f"{row['source_camera']}→"
            f"{row['target_camera']}"
        )
    )

    values.append(
        float(
            row[
                "fruit_balanced_mae"
            ]
        )
    )

ax.bar(
    labels,
    values
)

ax.axhline(
    session_null_metrics[
        "fruit_balanced_mae"
    ],
    linestyle="--",
    label="SessionExcluded_DummyMean"
)

ax.set_ylabel(
    "MAE fruit-balanced (días)"
)

ax.set_title(
    "Cross-sensor session-excluded"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.2
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "03_cross_sensor_session_excluded_mae.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 26. Guardar resultados centrales


In [ ]:
CROSS_STANDARD_OOF_PATH = (
    RESULTS /
    "kaki_cross_sensor_standard_oof.csv"
)

CROSS_SESSION_OOF_PATH = (
    RESULTS /
    "kaki_cross_sensor_session_excluded_oof.csv"
)

WITHIN_OOF_PATH = (
    RESULTS /
    "kaki_within_sensor_common_grid_oof.csv"
)

SESSION_NULL_PATH = (
    RESULTS /
    "kaki_session_excluded_dummy_mean_oof.csv"
)

MASTER_METRICS_PATH = (
    RESULTS /
    "kaki_cross_sensor_master_metrics.csv"
)

cross_standard_oof.to_csv(
    CROSS_STANDARD_OOF_PATH,
    index=False
)

cross_session_oof.to_csv(
    CROSS_SESSION_OOF_PATH,
    index=False
)

within_oof.to_csv(
    WITHIN_OOF_PATH,
    index=False
)

session_null_oof.to_csv(
    SESSION_NULL_PATH,
    index=False
)

comparison_metrics.to_csv(
    MASTER_METRICS_PATH,
    index=False
)

print("✅ Resultados guardados.")


# 27. Resumen científico automático


In [ ]:
def json_safe(
    obj
):

    if isinstance(
        obj,
        dict
    ):
        return {
            str(k):
                json_safe(
                    v
                )
            for k, v in obj.items()
        }

    if isinstance(
        obj,
        (
            list,
            tuple
        )
    ):
        return [
            json_safe(
                v
            )
            for v in obj
        ]

    if isinstance(
        obj,
        np.integer
    ):
        return int(
            obj
        )

    if isinstance(
        obj,
        np.floating
    ):
        return (
            None
            if np.isnan(
                obj
            )
            else float(
                obj
            )
        )

    if isinstance(
        obj,
        np.bool_
    ):
        return bool(
            obj
        )

    if isinstance(
        obj,
        np.ndarray
    ):
        return json_safe(
            obj.tolist()
        )

    return obj


summary = {
    "project":
        PROJECT_NAME,

    "notebook":
        NOTEBOOK_ID,

    "run_signature":
        RUN_SIGNATURE,

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "common_grid": {
        "min_nm":
            float(
                COMMON_GRID.min()
            ),

        "max_nm":
            float(
                COMMON_GRID.max()
            ),

        "step_nm":
            5.0,

        "n_features":
            int(
                len(
                    COMMON_GRID
                )
            ),
    },

    "directions":
        [
            "VIS_COR->VIS",
            "VIS->VIS_COR",
        ],

    "transfer_methods":
        TRANSFER_METHODS,

    "within_sensor_common_grid":
        within_metrics.to_dict(
            orient="records"
        ),

    "cross_sensor_standard":
        cross_standard_metrics.to_dict(
            orient="records"
        ),

    "cross_sensor_standard_selected_configs":
        cross_standard_selected.to_dict(
            orient="records"
        ),

    "cross_sensor_session_excluded":
        cross_session_metrics.to_dict(
            orient="records"
        ),

    "cross_sensor_session_selected_configs":
        cross_session_selected.to_dict(
            orient="records"
        ),

    "session_excluded_correct_null":
        session_null_metrics,

    "full_native_reference":
        full_reference.to_dict(
            orient="records"
        ),

    "paired_bootstrap":
        bootstrap_results.to_dict(
            orient="records"
        ),

    "interpretation": {
        "direct_transfer":
            (
                "No target-sensor calibration spectra are used "
                "for the selected direct method."
            ),

        "affine_transfer":
            (
                "If affine_* is selected, paired target/source spectra "
                "from development fruits are required for calibration."
            ),

        "session_excluded":
            (
                "Target day is excluded from predictor fitting and "
                "from affine calibration when affine transfer is used."
            ),
    },

    "limitations": [
        (
            "Both cameras belong to the same experimental campaign, so "
            "cross-sensor transfer is not external temporal validation."
        ),
        (
            "Affine calibration uses paired spectra from development fruits "
            "measured with both sensors and therefore is not zero-shot transfer."
        ),
        (
            "The paired cameras share fruit-day biology and campaign conditions; "
            "instrument transfer does not resolve storage-day/session confounding."
        ),
        (
            "External campaign and batch validation remains necessary."
        ),
    ],
}

SUMMARY_PATH = (
    RESULTS /
    "kaki_cross_sensor_transfer_calibration_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    )
)


# 28. Notas automáticas de revisión


In [ ]:
review_lines = [
    "KAKI — CROSS-SENSOR TRANSFER / CALIBRATION",
    "",
    f"Common grid: {COMMON_GRID.min():.0f}-{COMMON_GRID.max():.0f} nm",
    f"Grid step: 5 nm",
    f"Features: {len(COMMON_GRID)}",
    "",
    "WITHIN-SENSOR COMMON GRID:",
]

for _, r in within_metrics.iterrows():

    review_lines.append(
        (
            f"  {r['camera_type']}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"R2={r['r2']:.4f}, "
            f"±1d={r['within_1_day_accuracy']:.4f}"
        )
    )

review_lines.extend([
    "",
    "CROSS-SENSOR STANDARD:",
])

for _, r in cross_standard_metrics.iterrows():

    review_lines.append(
        (
            f"  {r['source_camera']}->{r['target_camera']}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"R2={r['r2']:.4f}, "
            f"±1d={r['within_1_day_accuracy']:.4f}"
        )
    )

review_lines.extend([
    "",
    "CROSS-SENSOR SESSION-EXCLUDED:",
])

for _, r in cross_session_metrics.iterrows():

    review_lines.append(
        (
            f"  {r['source_camera']}->{r['target_camera']}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"R2={r['r2']:.4f}, "
            f"improvement-null="
            f"{session_null_metrics['fruit_balanced_mae']-r['fruit_balanced_mae']:+.4f}"
        )
    )

review_lines.extend([
    "",
    (
        "Correct session-excluded null FB-MAE = "
        f"{session_null_metrics['fruit_balanced_mae']:.4f}"
    ),
    "",
    "METHOD STABILITY — STANDARD:",
])

for _, r in standard_method_stability.iterrows():

    review_lines.append(
        (
            f"  {r['source_camera']}->{r['target_camera']} | "
            f"{r['method']}: "
            f"{int(r['n_outer_folds_selected'])}/5 folds"
        )
    )

review_lines.extend([
    "",
    "Important:",
    (
        "Direct transfer requires no target-sensor calibration spectra. "
        "Affine transfer requires paired development measurements."
    ),
    (
        "Cross-sensor transfer is still within one campaign and must not be "
        "called external validation."
    ),
    (
        "Review this ZIP before deciding whether to move to trajectory models, "
        "formal manuscript synthesis, or deep-learning HSI."
    ),
])

REVIEW_PATH = (
    RESULTS /
    "CROSS_SENSOR_REVIEW_NOTES.txt"
)

REVIEW_PATH.write_text(
    "\n".join(
        review_lines
    )
    +
    "\n",
    encoding="utf-8"
)

print(
    REVIEW_PATH.read_text()
)
